# 08 — Minority-class oversampling

This experiment asks whether transparent random oversampling improves recognition of **`functional needs repair`**. It uses only the frozen development partition and its five fixed folds. Resampling happens *after* each fold split and only on that fold's training rows. The frozen local test is not opened, evaluated, or resampled.

The notebook also materialises reproducible, gzip-compressed development-training artefacts under `data/derived/08-minority-class-oversampling/`. Source rows remain traceable and the raw competition files are neither copied nor changed.

In [1]:
from pathlib import Path
import json
import platform
import sys

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / 'data'
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_partitioning import make_cross_validation, partition_modelling_data
from modelling_data import prepare_modelling_data
from model_evaluation import RANDOM_FOREST_ESTIMATORS
from oversampling import (
    BASELINE_NAME,
    CLASS_WEIGHTED_NAME,
    MINORITY_CLASS,
    OVERSAMPLED_NAME,
    OVERSAMPLING_SEED,
    comparison_table,
    evaluate_oversampling_strategies,
    write_oversampled_fold_artifacts,
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)
print({
    'python': platform.python_version(),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'scikit_learn': sklearn.__version__,
    'oversampling_seed': OVERSAMPLING_SEED,
})

{'python': '3.14.5', 'pandas': '3.0.5', 'numpy': '2.5.2', 'scikit_learn': '1.9.0', 'oversampling_seed': 20260821}


## 1. Define the goal and scope

**Users and decision:** the modelling team deciding whether an imbalance intervention should replace the unbalanced Random Forest training recipe.

**Success criteria:** improve minority-class recall and F1 while reporting macro-F1, balanced accuracy, ordinary accuracy, and all confusion matrices. A useful intervention should not buy minority recall through a severe collapse in overall discrimination.

**Scope:** one reproducible experiment on the frozen development folds. Competition submission and local-test evaluation are out of scope.

## 2. Gather the data

Load the immutable DrivenData files through the established modelling-data handoff, then apply the already-frozen split. The competition values are needed only to validate the existing source contract; they are not resampled or modelled here.

In [2]:
raw_original = pd.read_csv(DATA_DIR / 'TrainingSetValues.csv')
labels_original = pd.read_csv(DATA_DIR / 'TrainingSetLabels.csv')
raw_competition = pd.read_csv(DATA_DIR / 'TestSetValues.csv')

modelling_data = prepare_modelling_data(
    raw_original,
    labels_original,
    raw_competition,
)
partitioned_data = partition_modelling_data(modelling_data)
cross_validation = make_cross_validation(partitioned_data)

source_contract = pd.Series({
    'labelled source rows': len(modelling_data.original_ids),
    'development rows available to this experiment': len(partitioned_data.development_ids),
    'frozen local-test rows (ID metadata only)': len(partitioned_data.local_test_ids),
    'competition rows untouched': len(modelling_data.competition_ids),
    'development/local-test ID overlap': len(set(partitioned_data.development_ids) & set(partitioned_data.local_test_ids)),
    'development fingerprint': partitioned_data.development_fingerprint,
    'cross-validation fingerprint': partitioned_data.cross_validation_fingerprint,
    'frozen local-test fingerprint': partitioned_data.local_test_fingerprint,
}, name='value')
display(source_contract.to_frame())

,value
labelled source rows,59400
development rows available to this experiment,47520
frozen local-test rows (ID metadata only),11880
competition rows untouched,14850
development/local-test ID overlap,0
development fingerprint,c8a9e27264e4932140ffe9f19c897229230f9da00126f7...
cross-validation fingerprint,bd7da743e9b4498888fa9a7b4166f64266a5f86c5175b7...
frozen local-test fingerprint,6b5aff2b3b7d8dc3bc6ee1b36defabcfef312b03fef940...


## 3. Explore the data

Only development labels are explored. The table quantifies the imbalance that motivates the intervention; no frozen local-test labels are inspected.

In [3]:
development_distribution = (
    partitioned_data.y_development.value_counts()
    .rename('rows')
    .to_frame()
)
development_distribution['share'] = (
    development_distribution['rows'] / len(partitioned_data.y_development)
)
display(development_distribution)
print(
    f"'{MINORITY_CLASS}' has "
    f"{development_distribution.loc[MINORITY_CLASS, 'rows']:,} development rows "
    f"({development_distribution.loc[MINORITY_CLASS, 'share']:.1%})."
)

,rows,share
status_group,,
functional,25807,0.543077
non functional,18259,0.384238
functional needs repair,3454,0.072685


'functional needs repair' has 3,454 development rows (7.3%).


## 4. Clean and preprocess the data

The existing fold-fitted preprocessing pipeline handles missing numeric values, categorical encoding, and rare/unseen categories. No additional cleaning is needed for this experiment. Critically, preprocessing is learned separately inside each training fold; duplicated rows never influence validation preprocessing.

## 5. Select and engineer features

The established initial feature policy is reused unchanged so the comparison isolates the imbalance intervention. No new features are engineered here (**not applicable beyond reusing the frozen policy**). IDs and lineage columns remain metadata and never enter the model matrix.

## 6. Define the machine-learning task

This remains flat three-class classification. Oversampling does not assert that the labels are ordinal and does not alter their meaning; it changes only how often existing minority examples appear during fitting.

## 7. Partition the data

The local-test split and five validation folds were frozen upstream. The checks below prove that every development fold has disjoint training and validation memberships. Validation prevalence remains natural.

In [4]:
fold_rows = []
for fold_number, (training_positions, validation_positions) in enumerate(cross_validation.split(), start=1):
    assert not set(training_positions) & set(validation_positions)
    assert len(training_positions) + len(validation_positions) == len(partitioned_data.y_development)
    training_counts = partitioned_data.y_development.iloc[training_positions].value_counts()
    validation_counts = partitioned_data.y_development.iloc[validation_positions].value_counts()
    fold_rows.append({
        'validation_fold': fold_number,
        'training_rows': len(training_positions),
        'validation_rows': len(validation_positions),
        'training minority rows before resampling': training_counts[MINORITY_CLASS],
        'validation minority rows untouched': validation_counts[MINORITY_CLASS],
    })
fold_contract = pd.DataFrame(fold_rows).set_index('validation_fold')
display(fold_contract)

,training_rows,validation_rows,training minority rows before resampling,validation minority rows untouched
validation_fold,,,,
1,38016,9504,2764,690
2,38016,9504,2763,691
3,38016,9504,2763,691
4,38016,9504,2763,691
5,38016,9504,2763,691


## 8. Select and train candidate methods

Three versions of the project's conventional 300-tree Random Forest are compared:

1. the unbalanced baseline;
2. scikit-learn's balanced class weights; and
3. random oversampling with replacement, raising `functional needs repair` to the second-largest training class in that fold.

Random oversampling is deliberately transparent and keeps every generated row tied to a real source. SMOTE is not used: the predictors mix numeric and categorical concepts, and synthesising points in a one-hot or raw mixed-type space could create implausible water points without evidence that the extra assumption helps.

In [5]:
DERIVED_DIR = DATA_DIR / 'derived' / '08-minority-class-oversampling'
manifest = write_oversampled_fold_artifacts(
    partitioned_data,
    cross_validation,
    DERIVED_DIR,
    seed=OVERSAMPLING_SEED,
)
full_version = manifest['final_refit_development_version']
artifact_summary = pd.Series({
    'output directory': str(DERIVED_DIR),
    'full-development rows before': full_version['rows_before'],
    'full-development rows after': full_version['rows_after'],
    'compressed full-version MiB': full_version['file']['bytes'] / (1024 ** 2),
    'fold-specific versions': len(manifest['folds']),
    'local-test source-ID overlap': manifest['local_test_id_overlap'],
    'frozen local test opened or resampled': manifest['frozen_local_test_opened_or_resampled'],
}, name='value')
display(artifact_summary.to_frame())
display(pd.DataFrame([
    {
        'fold': fold['validation_fold'],
        'seed': fold['seed'],
        'training rows before': fold['training_rows_before'],
        'training rows after': fold['training_rows_after'],
        'validation rows untouched': fold['validation_rows_untouched'],
        'minority before': fold['counts_before'][MINORITY_CLASS],
        'minority after': fold['counts_after'][MINORITY_CLASS],
    }
    for fold in manifest['folds']
]).set_index('fold'))

,value
output directory,C:\_Source\Imperial-ML-AI-Live\Capstone\imperi...
full-development rows before,47520
full-development rows after,62325
compressed full-version MiB,4.172277
fold-specific versions,5
local-test source-ID overlap,0
frozen local test opened or resampled,False


,seed,training rows before,training rows after,validation rows untouched,minority before,minority after
fold,,,,,,
1,20260822,38016,49859,9504,2764,14607
2,20260823,38016,49861,9504,2763,14608
3,20260824,38016,49860,9504,2763,14607
4,20260825,38016,49860,9504,2763,14607
5,20260826,38016,49860,9504,2763,14607


In [6]:
print(f'Evaluating three {RANDOM_FOREST_ESTIMATORS}-tree Random Forest recipes over five frozen folds...')
evaluation = evaluate_oversampling_strategies(
    partitioned_data,
    cross_validation,
    seed=OVERSAMPLING_SEED,
)
comparison = comparison_table(evaluation)
display(comparison)

training_diagnostics = evaluation.diagnostics.groupby(level='strategy', sort=False).agg({
    'fit_rows': 'mean',
    'minority_fit_rows': 'mean',
    'replica_rows': 'mean',
    'elapsed_seconds': 'sum',
})
display(training_diagnostics)

Evaluating three 300-tree Random Forest recipes over five frozen folds...


Completed oversampling comparison fold 1: 38,016 original and 49,859 oversampled training rows.


Completed oversampling comparison fold 2: 38,016 original and 49,861 oversampled training rows.


Completed oversampling comparison fold 3: 38,016 original and 49,860 oversampled training rows.


Completed oversampling comparison fold 4: 38,016 original and 49,860 oversampled training rows.


Completed oversampling comparison fold 5: 38,016 original and 49,860 oversampled training rows.


,accuracy,balanced_accuracy,macro_f1,minority_recall,minority_f1
strategy,,,,,
unbalanced baseline,0.805913,0.677990,0.696396,0.368561,0.436110
balanced class weights,0.788931,0.706490,0.694220,0.499712,0.448233
random oversampling,0.797348,0.694653,0.695883,0.447310,0.442761


,fit_rows,minority_fit_rows,replica_rows,elapsed_seconds
strategy,,,,
unbalanced baseline,38016.0,2763.2,0.0,86.498177
balanced class weights,38016.0,2763.2,0.0,80.475059
random oversampling,49860.0,14607.2,11844.0,125.304784


## 9. Evaluate and interpret the results

Metrics are computed separately on every untouched validation fold, then averaged. The minority metrics make the intended trade-off visible; macro-F1 and balanced accuracy prevent the majority class from dominating the summary.

In [7]:
baseline = comparison.loc[BASELINE_NAME]
deltas = comparison.subtract(baseline, axis='columns')
display(deltas.rename_axis(index='change from unbalanced baseline'))

eligible = comparison.loc[
    comparison['macro_f1'].ge(baseline['macro_f1'] - 0.01)
    & comparison['accuracy'].ge(baseline['accuracy'] - 0.02)
]
selected_strategy = eligible['minority_f1'].idxmax()
print(f'Selected under the stated guardrails: {selected_strategy}')
print(
    f"Minority F1 change: {deltas.loc[selected_strategy, 'minority_f1']:+.4f}; "
    f"minority recall change: {deltas.loc[selected_strategy, 'minority_recall']:+.4f}; "
    f"macro-F1 change: {deltas.loc[selected_strategy, 'macro_f1']:+.4f}; "
    f"accuracy change: {deltas.loc[selected_strategy, 'accuracy']:+.4f}."
)

minority_by_fold = evaluation.fold_metrics.loc[:, ['minority_recall', 'minority_f1']].unstack(level='strategy')
display(minority_by_fold)

,accuracy,balanced_accuracy,macro_f1,minority_recall,minority_f1
change from unbalanced baseline,,,,,
unbalanced baseline,0.000000,0.000000,0.000000,0.000000,0.000000
balanced class weights,-0.016982,0.028500,-0.002176,0.131151,0.012124
random oversampling,-0.008565,0.016662,-0.000513,0.078749,0.006651


Selected under the stated guardrails: balanced class weights
Minority F1 change: +0.0121; minority recall change: +0.1312; macro-F1 change: -0.0022; accuracy change: -0.0170.


minority_recall                                                    minority_f1                      \
strategy        balanced class weights random oversampling unbalanced baseline balanced class weights random oversampling   
validation_fold                                                                                                             
1                             0.505797            0.455072            0.378261               0.454131            0.451799   
2                             0.473227            0.409551            0.342981               0.436000            0.418330   
3                             0.531114            0.471780            0.387844               0.465441            0.454355   
4                             0.505065            0.464544            0.370478               0.452953            0.460545   
5                             0.483357            0.435601            0.363242               0.432642            0.428775   

                                     
strategy        unbalanced baseline  
validation_fold                      
1                          0.445392  
2                          0.412892  
3                          0.449288  
4                          0.447552  
5                          0.425424

In [8]:
for strategy in (BASELINE_NAME, CLASS_WEIGHTED_NAME, OVERSAMPLED_NAME):
    print(f'\nAggregate out-of-fold confusion counts — {strategy}')
    display(evaluation.confusion_counts[strategy])
    print(f'Row-normalised recall — {strategy}')
    display(evaluation.confusion_recall[strategy])

assert set(evaluation.fold_confusions.index.get_level_values('validation_fold')) == {1, 2, 3, 4, 5}
assert all(
    int(evaluation.confusion_counts[strategy].to_numpy().sum())
    == len(partitioned_data.y_development)
    for strategy in (BASELINE_NAME, CLASS_WEIGHTED_NAME, OVERSAMPLED_NAME)
)
print('All five fold-level confusion matrices are retained and every strategy predicts each development row exactly once.')


Aggregate out-of-fold confusion counts — unbalanced baseline


predicted,functional,functional needs repair,non functional
actual,,,
functional,22618,775,2414
functional needs repair,1639,1273,542
non functional,3518,335,14406


Row-normalised recall — unbalanced baseline


predicted,functional,functional needs repair,non functional
actual,,,
functional,0.876429,0.030031,0.093541
functional needs repair,0.474522,0.368558,0.156920
non functional,0.192672,0.018347,0.788981



Aggregate out-of-fold confusion counts — balanced class weights


...


Aggregate out-of-fold confusion counts — random oversampling


predicted,functional,functional needs repair,non functional
actual,,,
functional,22092,1392,2323
functional needs repair,1440,1545,469
non functional,3421,585,14253


Row-normalised recall — random oversampling


predicted,functional,functional needs repair,non functional
actual,,,
functional,0.856047,0.053939,0.090014
functional needs repair,0.416908,0.447307,0.135785
non functional,0.187360,0.032039,0.780601


All five fold-level confusion matrices are retained and every strategy predicts each development row exactly once.


## 10. Deploy and iterate

**Deployment is not applicable:** this experiment neither opens the frozen local test nor creates a competition submission. The selected intervention should next be treated as a candidate recipe and evaluated once on the frozen local test only when model selection is genuinely complete.

The full-development oversampled file is a **development-refit/training variant** for fitting before a future sealed local-test evaluation. It is not a true final all-labelled refit because local-test rows remain excluded. It must not be used for cross-validation because it contains every development fold; the five fold-specific variants are the leakage-safe versions for repeatable development evaluation.

In [9]:
manifest_path = DERIVED_DIR / 'manifest.json'
manifest_from_disk = json.loads(manifest_path.read_text(encoding='utf-8'))
full_artifact_path = DERIVED_DIR / manifest_from_disk['final_refit_development_version']['file']['path']
assert manifest_from_disk['local_test_id_overlap'] == 0
assert manifest_from_disk['frozen_local_test_opened_or_resampled'] is False
assert full_artifact_path.is_file()
assert len(manifest_from_disk['folds']) == 5
assert all(
    (DERIVED_DIR / fold['files']['values']['path']).is_file()
    and (DERIVED_DIR / fold['files']['labels']['path']).is_file()
    and (DERIVED_DIR / fold['files']['lineage']['path']).is_file()
    for fold in manifest_from_disk['folds']
)

lifecycle = pd.DataFrame([
    (1, 'Define the goal and scope', 'Complete', 'Decision and guardrail metrics stated'),
    (2, 'Gather the data', 'Complete', 'Immutable source contract and frozen partitions loaded'),
    (3, 'Explore the data', 'Complete', 'Development-only class imbalance quantified'),
    (4, 'Clean and preprocess the data', 'Complete', 'Existing preprocessing fitted within folds'),
    (5, 'Select and engineer features', 'Not applicable: no new policy', 'Existing feature policy held fixed'),
    (6, 'Define the machine-learning task', 'Complete', 'Flat three-class classification'),
    (7, 'Partition the data', 'Complete', 'Frozen local split and five folds preserved'),
    (8, 'Select and train candidate methods', 'Complete', 'Baseline, class weights and random oversampling compared'),
    (9, 'Evaluate and interpret the results', 'Complete', 'Five metrics and fold/aggregate confusion matrices retained'),
    (10, 'Deploy and iterate', 'Not applicable: no deployment', 'Materialised refit candidate; local test remains sealed'),
], columns=['step', 'lifecycle stage', 'status', 'evidence']).set_index('step')
display(lifecycle)
print(f'Verified manifest: {manifest_path}')
print(f'Verified full-development artefact: {full_artifact_path}')

,lifecycle stage,status,evidence
step,,,
1,Define the goal and scope,Complete,Decision and guardrail metrics stated
2,Gather the data,Complete,Immutable source contract and frozen partition...
3,Explore the data,Complete,Development-only class imbalance quantified
4,Clean and preprocess the data,Complete,Existing preprocessing fitted within folds
5,Select and engineer features,Not applicable: no new policy,Existing feature policy held fixed
6,Define the machine-learning task,Complete,Flat three-class classification
7,Partition the data,Complete,Frozen local split and five folds preserved
8,Select and train candidate methods,Complete,"Baseline, class weights and random oversamplin..."
9,Evaluate and interpret the results,Complete,Five metrics and fold/aggregate confusion matr...


Verified manifest: C:\_Source\Imperial-ML-AI-Live\Capstone\imperial-capstone\stage-1-pump-it-up\data\derived\08-minority-class-oversampling\manifest.json
Verified full-development artefact: C:\_Source\Imperial-ML-AI-Live\Capstone\imperial-capstone\stage-1-pump-it-up\data\derived\08-minority-class-oversampling\development-training-oversampled-with-lineage.csv.gz
